##### Content of this Notebook:
- ###### Transformation 24: When Otherwise
- ###### Transformation 25: Joins (Inner, Left, Right, Anti)
- ###### Transformation 26: Windows Function (Row_Number(),Rank(),Dense_Rank(),CumSum())
- ###### Transformation 27: User Defined Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = spark.read.table('workspace.default.big_mart_sales')
df.display()

###### Transformation 24: When Otherwise

In [0]:
df_vegflag = df.withColumn('Veg_Flag',when(col('Item_Type')=='Meat',"Non-Veg").otherwise("Veg"))
df_vegflag.display()

In [0]:
df_vegflag.withColumn('Premium_Cat_Flag',when((col('Veg_Flag')=='Veg') & (col('Item_MRP')>=100),'Premium Veg')\
                                        .when((col('Veg_Flag')=='Veg') & (col('Item_MRP')<100),'Budget Veg')\
                                        .otherwise('Non-Veg'))\
                                        .display()

#### Transformation 25: JOINS

###### Innner
![image_1784214737599.png](./image_1784214737599.png "image_1784214737599.png")

In [0]:
data1 = [('1','Gaur','D01'),
         ('2','Kit','D02'),
         ('3','Sam','D03'),
         ('4','Tim','D03'),
         ('5','Aman','D05'),]
schema1 = 'Emp_ID STRING, Emp_Name STRING, Emp_Dept STRING'
df1 = spark.createDataFrame(data=data1, schema = schema1)
df1.display()
data2 = [('D01','HR'),
         ('D02','Marketing'),
         ('D03','Account'),
         ('D04','IT'),
         ('D05','Finance'),]
schema2 = 'Dept_ID STRING, Dept_Name STRING'
df2 = spark.createDataFrame(data=data2, schema = schema2)
df2.display()

In [0]:
df1.join(df2, df1['Emp_Dept']==df2['Dept_ID'],'inner').display()

###### Left Join
![image_1784215633341.png](./image_1784215633341.png "image_1784215633341.png")

In [0]:
data1 = [('1','Gaur','D01'),
         ('2','Kit','D02'),
         ('3','Sam','D03'),
         ('4','Tim','D03'),
         ('5','Aman','D05'),
         ('6','Nat','D06')]
schema1 = 'Emp_ID STRING, Emp_Name STRING, Emp_Dept STRING'
df1 = spark.createDataFrame(data=data1, schema = schema1)
df1.display()
data2 = [('D01','HR'),
         ('D02','Marketing'),
         ('D03','Account'),
         ('D04','IT'),
         ('D05','Finance'),]
schema2 = 'Dept_ID STRING, Dept_Name STRING'
df2 = spark.createDataFrame(data=data2, schema = schema2)
df2.display()

In [0]:
df1.join(df2, df1['Emp_Dept']==df2['Dept_ID'],'left').display()

###### Right Join:
![image_1784215661788.png](./image_1784215661788.png "image_1784215661788.png")

In [0]:
df1.join(df2, df1['Emp_Dept']==df2['Dept_ID'],'right').display()

###### Anti Join: The join through which we can find the records of DF1 which is NOT available in DF2
![image_1784215931686.png](./image_1784215931686.png "image_1784215931686.png")

In [0]:
df1.join(df2, df1['Emp_Dept']==df2['Dept_ID'],'anti').display()

#### Transformation 26: Windows Function
![image_1784272675162.png](./image_1784272675162.png "image_1784272675162.png")

In [0]:
from pyspark.sql.window import Window

###### Row_Number()

In [0]:
df.withColumn('RowNumCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

###### Rank()

In [0]:
df.withColumn('RowNumCol', row_number().over(Window.orderBy(col('Item_Identifier').desc())))\
  .withColumn('RankCol', rank().over(Window.orderBy('Item_Identifier')))\
  .display()

###### DenseRank()

In [0]:
df.withColumn('RowNumCol', row_number().over(Window.orderBy(col('Item_Identifier').desc())))\
  .withColumn('RankCol', rank().over(Window.orderBy(col('Item_Identifier').desc())))\
  .withColumn('DenseRankCol', dense_rank().over(Window.orderBy(col('Item_Identifier').desc())))\
  .display()

###### Cumulative Sum

In [0]:
df.withColumn('CumSum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('CumSum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.currentRow,Window.unboundedFollowing))).display()

#### Tranformation 27: User Defined Functions